In [1]:
import sys
import os
sys.path.append(os.path.abspath('../src')) ## Add the source directory to the Python path
import torch
import numpy as np
import pandas as pd
import json
import time
from collections import defaultdict
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import RobertaForSequenceClassification, RobertaTokenizer
from google import genai
from dotenv import load_dotenv
from data_loader import load_edos_data, TASK_A_ID2LABEL, TASK_B_ID2LABEL
DATA_DIR='../data/'
RESULTS_DIR='../results/'
MODELS_DIR='../models/'
KG_DIR='../kg/'
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

c:\Users\Ashwin Nair\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
load_dotenv('../.env') ## Load environment variables from .env file
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY not found in .env file')
client=genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL='gemini-2.5-flash-lite'
time.sleep(3)
response = client.models.generate_content(model=GEMINI_MODEL, contents='Reply with OK only.')
print(f'Gemini API working: {response.text.strip()}')

Gemini API working: OK


In [3]:
def load_roberta(task: str): ## Load a pre-trained RoBERTa model for a specific task
    model_path = os.path.join(MODELS_DIR, f'roberta_task_{task}')
    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Model not found at {model_path}. Run previous notebook (02_baseline1_roberta.ipynb).')
    tokenizer=RobertaTokenizer.from_pretrained('roberta-base')
    model=RobertaForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()
    print(f'Loaded RoBERTa Task {task} from {model_path}')
    return model, tokenizer

roberta_a, tokenizer_a=load_roberta('A')  ## Load RoBERTa model for Task A
roberta_b, tokenizer_b=load_roberta('B') ## Load RoBERTa model for Task B

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3618.81it/s]


Loaded RoBERTa Task A from ../models/roberta_task_A


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3291.27it/s]


Loaded RoBERTa Task B from ../models/roberta_task_B


The Sexism KG was built from 3,398 sexist training posts (Task B training split) using Gemini to extract semantic triples.

**Construction pipeline:**
- Loaded all sexist training posts from EDOS Task B training split
- For each post, prompted Gemini to extract 1-3 subject-relation-object triples using a fixed schema of 6 relation types aligned to EDOS Task B categories
- Saved raw KG: **6,270 triples** across 6 relations

**Cleaning pipeline (applied in Notebook 04):**
- Normalized pronouns: she/her -> women, he/him -> men (1,187 triples updated)
- Dropped ambiguous pronoun entities: they, you, it, we, this, that (224 triples removed)
- Removed ASSIGNED_TO_ROLE relation entirely - only 25.3% alignment with prejudiced discussions category, worse than random (760 triples removed)
- Result: **5,286 clean triples** across 5 relations

**Relation alignment after cleaning:**
- THREATENED_WITH -> threats: 70.6%
- EXPRESSED_ANIMOSITY_TOWARDS -> animosity: 70.8%
- FRAMED_AS_INFERIOR -> derogation: 58.5%
- STEREOTYPED_AS -> derogation: 55.6%
- IDEOLOGICALLY_DISCREDITED -> prejudiced discussions: 52.0%

In [4]:
kg_path=os.path.join(KG_DIR, 'sexism_kg_clean.json') ## Load the cleaned knowledge graph
with open(kg_path, 'r') as f: ## Open the JSON file
    kg=json.load(f)
all_triples=kg['triples']
entity_index=kg['entity_index']
relation_index=kg['relation_index']
print(f'Clean KG loaded:')
print(f'Total triples:{len(all_triples)}')
print(f'Unique entities:{len(entity_index)}')
print(f'Unique relations:{len(relation_index)}')

Clean KG loaded:
Total triples:5286
Unique entities:5570
Unique relations:5


In [ ]:
# Confidence thresholds
CONFIDENCE_THRESHOLD_A=0.80 
CONFIDENCE_THRESHOLD_B=0.60
# Category to relevant relations
# Only retrieve relations aligned with RoBERTa's predicted category
CATEGORY_TO_RELATIONS = { 
    '1. threats, plans to harm and incitement': ['THREATENED_WITH'],
    '2. derogation': ['STEREOTYPED_AS', 'FRAMED_AS_INFERIOR'],
    '3. animosity': ['EXPRESSED_ANIMOSITY_TOWARDS'],
    '4. prejudiced discussions': ['IDEOLOGICALLY_DISCREDITED']
}
# Entity normalization
ENTITY_NORMALIZER = { 
    'she': 'women', 'her': 'women', 'woman': 'women',
    'a woman': 'women', 'female': 'women', 'females': 'women',
    'girl': 'women', 'girls': 'women',
    'he': 'men', 'him': 'men'
}
# Pronouns to skip
PRONOUNS_TO_SKIP = {
    'they', 'you', 'it', 'we', 'i', 'this', 'that',
    'them', 'their', 'a', 'an', 'the', 'is', 'are',
    'was', 'were', 'be', 'been'
}
MAX_TRIPLES_TO_RETRIEVE=3 
print('Configuration loaded.')
print(f'Confidence thresholds: Task A={CONFIDENCE_THRESHOLD_A} | Task B={CONFIDENCE_THRESHOLD_B}')
print(f'Max triples per post: {MAX_TRIPLES_TO_RETRIEVE}')

Configuration loaded.
Confidence thresholds: Task A=0.8 | Task B=0.6
Max triples per post: 3


In [ ]:
def roberta_predict(text: str, model, tokenizer, id2label: dict) -> tuple: # Predict the category of a text using a RoBERTa model
    encoding=tokenizer(text, truncation=True, padding='max_length',max_length=128, return_tensors='pt').to(DEVICE) ## Move the encoding to the specified device
    with torch.no_grad(): ## Disable gradient calculation for inference 
        outputs=model(**encoding) ## Get model outputs
        probs=torch.softmax(outputs.logits, dim=-1).squeeze() ## Compute probabilities using softmax

    pred_id=torch.argmax(probs).item() ## Get the predicted class ID
    confidence=probs[pred_id].item() ## Get the confidence score for the predicted class
    label=id2label[pred_id] ## Get the label corresponding to the predicted class ID
    all_probs={id2label[i]: round(probs[i].item(), 4) for i in range(len(probs))} 
    return label, confidence, all_probs

In [7]:
def extract_entities_from_post(text: str) -> list: ## Extract entities from a post, normalize them, and return a list of unique entities
    words=text.lower().split()
    entities=set() ## Initialize a set to store unique entities
    for word in words: 
        word = word.strip('.,!?;:"\'()[]')
        if len(word) < 3 or word in PRONOUNS_TO_SKIP: ## Skip short words and pronouns
            continue
        normalized = ENTITY_NORMALIZER.get(word, word) ## Normalize the entity using the ENTITY_NORMALIZER dictionary
        entities.add(normalized)
    # Always check for key entities
    text_lower=text.lower()
    for phrase in ['women', 'men', 'feminists', 'feminist', 'females', 'girls']: ## Check for key entities
        if phrase in text_lower:
            entities.add(phrase)

    return list(entities)

In [8]:
def retrieve_triples(text: str, roberta_category: str, task: str) -> list: 
    """
    Retrieve KG triples using entity + category-filtered relation matching.
    Task A: any relation
    Task B: only relations relevant to RoBERTa's predicted category
    """
    entities = extract_entities_from_post(text)
    if task == 'B':
        relevant_relations = set(CATEGORY_TO_RELATIONS.get(roberta_category, list(relation_index.keys()))) ## Get relevant relations for the predicted category
    else:
        relevant_relations = set(relation_index.keys())
    retrieved=[]
    seen_triples=set()
    for entity in entities:
        candidate_indices=entity_index.get(entity, []) ## Get candidate triple indices for the entity
        for idx in candidate_indices:
            triple=all_triples[idx]
            if triple['relation'] not in relevant_relations: ## Skip triples whose relation is not relevant to the predicted category
                continue
            key = (triple['subject'], triple['relation'], triple['object'])
            if key in seen_triples: ## Skip duplicate triples
                continue
            seen_triples.add(key)
            retrieved.append(triple)
    # Prioritize triples whose relation exactly matches predicted category
    if task=='B' and roberta_category in CATEGORY_TO_RELATIONS: ## Check if the task is B and the predicted category has relevant relations
        primary = CATEGORY_TO_RELATIONS[roberta_category]
        retrieved.sort(key=lambda t: 0 if t['relation'] in primary else 1)

    return retrieved[:MAX_TRIPLES_TO_RETRIEVE] ## Return only the top N triples based on MAX_TRIPLES_TO_RETRIEVE

In [9]:
def format_kg_context(triples: list) -> str: ## Format retrieved triples into readable text for the LLM prompt
    if not triples:
        return "No relevant knowledge found."
    lines = [f"- ({t['subject']}, {t['relation']}, {t['object']})" for t in triples]
    return '\n'.join(lines)

In [10]:
TASK_A_PROMPT_KG = """You are an expert in detecting sexist content on social media.

A classifier analysed this post but is NOT confident in its prediction.

Post: "{text}"

Classifier prediction: {prediction} (confidence: {confidence:.1%}) — LOW CONFIDENCE
Classifier probabilities: {probs}

Relevant knowledge from similar training posts:
{kg_context}

Using both the post content and the knowledge above, determine the correct label.
Only override the classifier if you have clear evidence it is wrong.

Respond with ONLY one of these two labels (no explanation):
not sexist
sexist"""


TASK_B_PROMPT_KG = """You are an expert in categorising sexist content on social media.

A classifier analysed this post but is NOT confident about its category.

Post: "{text}"

Classifier prediction: {prediction} (confidence: {confidence:.1%}) — LOW CONFIDENCE
Classifier probabilities: {probs}

Relevant knowledge from similar training posts:
{kg_context}

Categories:
- threats — direct or indirect threats of violence or harm against women
- derogation — insults, slurs, or portraying women as inferior
- animosity — hostility, contempt, or hatred towards women or feminists
- prejudiced discussions — ideological claims about gender roles or women's place in society

Using both the post content and the knowledge above, determine the correct category.
Only override the classifier if you have clear evidence it is wrong.

Respond with ONLY one of these exact words or phrases (no explanation, no numbers):
threats
derogation
animosity
prejudiced discussions"""

In [11]:
TASK_B_SHORT_TO_FULL = {
    'threats': '1. threats, plans to harm and incitement',
    'threat':'1. threats, plans to harm and incitement',
    'derogation':'2. derogation',
    'animosity':'3. animosity',
    'animosty':'3. animosity',
    'prejudiced discussions':'4. prejudiced discussions',
    'prejudiced':'4. prejudiced discussions',
}

In [12]:
def llm_reason_with_kg(text: str, roberta_label: str, confidence: float,all_probs: dict, kg_triples: list,task: str, max_retries: int = 3):
    """
    Call Gemini with post + RoBERTa prediction + KG triples.
    Returns final label or None if all retries fail.
    """
    kg_context = format_kg_context(kg_triples) ## Format the retrieved triples into a readable string for the LLM prompt
    if task=='A': ### If the task is A, use the Task A prompt
        prompt=TASK_A_PROMPT_KG.format(text=text, prediction=roberta_label,confidence=confidence, probs=all_probs,kg_context=kg_context)
        valid_labels={'not sexist', 'sexist'}
    else: ### If the task is B, use the Task B prompt
        prompt=TASK_B_PROMPT_KG.format(text=text, prediction=roberta_label,confidence=confidence, probs=all_probs,kg_context=kg_context)
        valid_labels={
            '1. threats, plans to harm and incitement',
            '2. derogation', '3. animosity',
            '4. prejudiced discussions'
        }
    for attempt in range(max_retries): ### Retry loop for calling the Gemini model
        try:
            response=client.models.generate_content(model=GEMINI_MODEL, contents=prompt) ## Call the Gemini model with the prompt
            if response is None or not hasattr(response, 'text') or response.text is None: ### Check if the response is empty or invalid
                raise ValueError('Empty response from Gemini')
            llm_label=response.text.strip().lower()
            if llm_label in valid_labels:
                return llm_label
            if task=='B': ## If the task is B, check for short forms of the labels
                if llm_label in TASK_B_SHORT_TO_FULL:
                    return TASK_B_SHORT_TO_FULL[llm_label]
                for short, full in TASK_B_SHORT_TO_FULL.items():
                    if short in llm_label:
                        return full
            if task == 'A': ## If the task is A, check if any valid label is a substring of the LLM response
                for valid in valid_labels:
                    if valid in llm_label:
                        return valid
            print(f'Unrecognised response: "{llm_label}" - skipping')
            return None

        except Exception as e: ## Handle exceptions during the Gemini API call
            error_str = str(e)
            if '429' in error_str or '503' in error_str: ## If the error is a rate limit or server busy error, wait longer before retrying
                wait = 60 * (attempt + 1)
                print(f'Server busy - waiting {wait}s (retry {attempt+1}/{max_retries})')
                time.sleep(wait)
            elif attempt < max_retries-1:
                print(f'Attempt {attempt+1} failed: {e} - retrying in 15s')
                time.sleep(15)
            else:
                print(f'Failed after {max_retries} attempts: {e}')

    return None